# Experiment 6/7 — Bagging, Boosting, and Stacked Ensemble Models

Generic `assn6_experiment()` function — works on any tabular classification dataset (just pass a dataframe and target column). Dataset used here: the **Wisconsin Diagnostic Breast Cancer Dataset** (569 samples, 30 features, loaded directly from `sklearn.datasets` — no separate CSV needed).

Implements Bagging (Decision Tree base), Boosting (AdaBoost + Gradient Boosting), and a Stacked Ensemble (SVM + Naive Bayes + Decision Tree base learners, Logistic Regression meta-learner), each hyperparameter-tuned with 5-fold cross-validation, then compared on the held-out test set.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    ConfusionMatrixDisplay, roc_curve, roc_auc_score
)

## `assn6_experiment()`

Broken into numbered steps, same style as the earlier assignments — every choice is commented so you can explain it in an exam:

STEP 1: Preprocess (split + scale)
STEP 2: EDA plots
STEP 3: Bagging Classifier (tuned)
STEP 4: Boosting — AdaBoost + Gradient Boosting (tuned)
STEP 5: Stacked Ensemble (tuned base learners + meta-learner)
STEP 6: Final test-set evaluation + ROC curves
STEP 7: Bias–variance check (train vs test accuracy)

In [ ]:
def assn6_experiment(
    df,
    target_column,
    test_size=0.20,
    cv_folds=5,
    random_state=42,
    bagging_param_grid=None,
    ada_param_grid=None,
    gb_param_grid=None
):
    """
    Assignment 6/7 — Bagging, Boosting, and Stacked Ensemble pipeline.
    Works on ANY tabular classification dataset (df + target_column).

    STEP 1: Preprocess (split + scale)
    STEP 2: EDA plots
    STEP 3: Bagging Classifier (tuned)
    STEP 4: Boosting -- AdaBoost + Gradient Boosting (tuned)
    STEP 5: Stacked Ensemble (tuned base learners + meta-learner)
    STEP 6: Final test-set evaluation + ROC curves
    STEP 7: Bias-variance check (train vs test accuracy)
    """

    if bagging_param_grid is None:
        bagging_param_grid = {
            "n_estimators": [10, 50, 100],
            "max_samples": [0.5, 0.7, 1.0],
            "max_features": [0.5, 0.7, 1.0]
        }
    if ada_param_grid is None:
        ada_param_grid = {"n_estimators": [50, 100, 150], "learning_rate": [0.01, 0.1, 1.0]}
    if gb_param_grid is None:
        gb_param_grid = {"n_estimators": [50, 100, 150], "learning_rate": [0.01, 0.1, 1.0], "max_depth": [2, 3, 4]}

    # ==============================================================
    # STEP 1: PREPROCESS
    # ==============================================================
    print("Shape:", df.shape)
    print("Missing values:", df.isnull().sum().sum())
    print("Class balance:\n", df[target_column].value_counts())

    X = df.drop(columns=[target_column])
    y = df[target_column]

    # Stratified split -> keeps the malignant/benign ratio consistent across
    # train and test, same reasoning as assn1's classification split.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    # Standardize -> SVM (used inside the Stacked Ensemble) is distance-based
    # and sensitive to feature scale, same reasoning as KNN in Experiment 2.
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    print("Train shape:", X_train_s.shape, "Test shape:", X_test_s.shape)

    # ==============================================================
    # STEP 2: EDA
    # ==============================================================
    plt.figure(figsize=(6, 5))
    sns.countplot(x=y)
    plt.title("Class Distribution")
    plt.show()

    plt.figure(figsize=(16, 13))
    sns.heatmap(df.corr(), cmap="coolwarm")
    plt.title("Correlation Heatmap")
    plt.show()

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, col in zip(axes.flat, df.columns[:6]):
        df[col].hist(ax=ax, bins=20)
        ax.set_title(col)
    plt.tight_layout()
    plt.show()

    # ==============================================================
    # STEP 3: BAGGING
    # ==============================================================
    # Bagging trains many Decision Trees, each on a random bootstrap sample
    # of the training data (with replacement), and averages their votes.
    # Because each tree sees slightly different data, their individual
    # mistakes tend to cancel out when combined -> this is what REDUCES
    # VARIANCE (a single deep decision tree overfits easily; many bagged
    # trees, averaged, don't).
    bagging_grid = GridSearchCV(
        BaggingClassifier(estimator=DecisionTreeClassifier(random_state=random_state), random_state=random_state),
        bagging_param_grid, cv=cv_folds, scoring="accuracy", n_jobs=-1
    )
    start = time.time()
    bagging_grid.fit(X_train_s, y_train)
    bagging_time = time.time() - start

    cvres = pd.DataFrame(bagging_grid.cv_results_)
    bagging_table = cvres[["param_n_estimators", "param_max_samples", "param_max_features", "mean_test_score"]].copy()
    bagging_table = bagging_table.sort_values("mean_test_score", ascending=False).reset_index(drop=True)
    bagging_table["mean_test_score"] = bagging_table["mean_test_score"] * 100
    bagging_table.columns = ["n_estimators", "max_samples", "max_features", "Avg CV Accuracy (%)"]
    print("\nBagging Hyperparameter Evaluation (top 8)")
    display(bagging_table.head(8))

    best_bagging = bagging_grid.best_estimator_
    bagging_f1 = cross_val_score(best_bagging, X_train_s, y_train, cv=cv_folds, scoring="f1").mean()
    print("Bagging best params:", bagging_grid.best_params_)
    print("Bagging best CV accuracy: {:.2f}%   Bagging best CV F1: {:.4f}".format(
        bagging_grid.best_score_ * 100, bagging_f1))

    # ==============================================================
    # STEP 4: BOOSTING -- AdaBoost + Gradient Boosting
    # ==============================================================
    # Boosting trains models SEQUENTIALLY: each new model focuses extra
    # attention on the examples the previous models got wrong (AdaBoost
    # reweights misclassified samples; Gradient Boosting fits each new
    # model to the previous model's residual errors). This is what REDUCES
    # BIAS -- a single weak learner (e.g. a shallow tree) underfits, but a
    # sequence of them, each correcting the last, can capture a much more
    # complex pattern.
    ada_grid = GridSearchCV(AdaBoostClassifier(random_state=random_state), ada_param_grid, cv=cv_folds, scoring="accuracy", n_jobs=-1)
    start = time.time()
    ada_grid.fit(X_train_s, y_train)
    ada_time = time.time() - start

    gb_grid = GridSearchCV(GradientBoostingClassifier(random_state=random_state), gb_param_grid, cv=cv_folds, scoring="accuracy", n_jobs=-1)
    start = time.time()
    gb_grid.fit(X_train_s, y_train)
    gb_time = time.time() - start

    ada_f1 = cross_val_score(ada_grid.best_estimator_, X_train_s, y_train, cv=cv_folds, scoring="f1").mean()
    gb_f1 = cross_val_score(gb_grid.best_estimator_, X_train_s, y_train, cv=cv_folds, scoring="f1").mean()

    boosting_table = pd.DataFrame([
        {"Model": "AdaBoost", "n_estimators": ada_grid.best_params_["n_estimators"],
         "learning_rate": ada_grid.best_params_["learning_rate"],
         "Avg CV Accuracy (%)": ada_grid.best_score_ * 100, "Avg CV F1 Score": ada_f1},
        {"Model": "Gradient Boosting", "n_estimators": gb_grid.best_params_["n_estimators"],
         "learning_rate": gb_grid.best_params_["learning_rate"],
         "Avg CV Accuracy (%)": gb_grid.best_score_ * 100, "Avg CV F1 Score": gb_f1},
    ])
    print("\nBoosting Hyperparameter Evaluation")
    display(boosting_table)

    # pick whichever boosting variant scored higher on CV accuracy to carry forward
    best_boosting_name = "AdaBoost" if ada_grid.best_score_ >= gb_grid.best_score_ else "Gradient Boosting"
    best_boosting = ada_grid.best_estimator_ if best_boosting_name == "AdaBoost" else gb_grid.best_estimator_
    print("Best boosting variant:", best_boosting_name)

    # ==============================================================
    # STEP 5: STACKED ENSEMBLE
    # ==============================================================
    # Stacking trains several DIFFERENT ("heterogeneous") base models --
    # here SVM, Naive Bayes, and a Decision Tree -- each of which makes
    # mistakes in different places because they work on different
    # assumptions (margins, probability, splits). Instead of a simple vote,
    # a META-LEARNER (Logistic Regression) is trained on the base models'
    # OUTPUT PREDICTIONS to learn the best way to combine them. This can
    # outperform any single base model because it exploits their diversity.
    base_estimators = [
        ("svm", SVC(probability=True, random_state=random_state)),
        ("nb", GaussianNB()),
        ("dt", DecisionTreeClassifier(random_state=random_state))
    ]
    stack = StackingClassifier(
        estimators=base_estimators,
        final_estimator=LogisticRegression(max_iter=5000, random_state=random_state),
        cv=cv_folds
    )
    start = time.time()
    stack.fit(X_train_s, y_train)
    stack_time = time.time() - start

    stack_cv_acc = cross_val_score(stack, X_train_s, y_train, cv=cv_folds, scoring="accuracy").mean()
    stack_cv_f1 = cross_val_score(stack, X_train_s, y_train, cv=cv_folds, scoring="f1").mean()

    stack_table = pd.DataFrame([{
        "Base Models": "SVM, Naive Bayes, Decision Tree",
        "Meta Learner": "Logistic Regression",
        "Avg CV Accuracy (%)": stack_cv_acc * 100,
        "Avg CV F1 Score": stack_cv_f1
    }])
    print("\nStacked Ensemble Evaluation")
    display(stack_table)

    # ==============================================================
    # STEP 6: FINAL TEST SET EVALUATION
    # ==============================================================
    final_models = {
        "Bagging": best_bagging,
        f"Boosting ({best_boosting_name})": best_boosting,
        "Stacked Ensemble": stack
    }

    results = []
    roc_data = {}
    for name, model in final_models.items():
        pred = model.predict(X_test_s)
        prob = model.predict_proba(X_test_s)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, prob)
        auc = roc_auc_score(y_test, prob)
        roc_data[name] = (fpr, tpr, auc)

        results.append({
            "Model": name,
            "Accuracy": accuracy_score(y_test, pred) * 100,
            "Precision": precision_score(y_test, pred),
            "Recall": recall_score(y_test, pred),
            "F1 Score": f1_score(y_test, pred),
            "AUC": auc
        })

        ConfusionMatrixDisplay.from_predictions(y_test, pred)
        plt.title(f"Confusion Matrix - {name}")
        plt.show()

    results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False).reset_index(drop=True)
    print("\nPerformance Comparison of Ensemble Models")
    display(results_df)

    # ROC curves -> shows how well each model ranks positive vs negative
    # cases across every possible decision threshold, not just the default
    # 0.5 cutoff. A curve closer to the top-left corner (AUC closer to 1.0)
    # means better separation between the two classes.
    plt.figure(figsize=(7, 6))
    for name, (fpr, tpr, auc) in roc_data.items():
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.4f})")
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curves - Ensemble Models")
    plt.legend()
    plt.show()

    # ==============================================================
    # STEP 7: BIAS-VARIANCE CHECK (train vs test accuracy)
    # ==============================================================
    # A big gap between training accuracy and test accuracy is a classic
    # sign of overfitting (high variance) -- the model memorized the
    # training data instead of learning a generalizable pattern.
    train_acc, test_acc = [], []
    for name, model in final_models.items():
        train_acc.append(accuracy_score(y_train, model.predict(X_train_s)) * 100)
        test_acc.append(accuracy_score(y_test, model.predict(X_test_s)) * 100)

    x = np.arange(len(final_models))
    plt.figure(figsize=(8, 5))
    plt.bar(x - 0.2, train_acc, width=0.4, label="Train Accuracy")
    plt.bar(x + 0.2, test_acc, width=0.4, label="Test Accuracy")
    plt.xticks(x, list(final_models.keys()), rotation=10)
    plt.ylabel("Accuracy (%)")
    plt.title("Train vs Test Accuracy (Bias-Variance Check)")
    plt.legend()
    plt.show()

    return {
        "bagging_table": bagging_table,
        "boosting_table": boosting_table,
        "stack_table": stack_table,
        "results_df": results_df,
        "best_bagging": best_bagging,
        "best_boosting": best_boosting,
        "best_boosting_name": best_boosting_name,
        "stack": stack,
        "scaler": scaler,
        "X_train": X_train_s,
        "X_test": X_test_s,
        "y_train": y_train,
        "y_test": y_test
    }

### Usage

The Wisconsin Breast Cancer dataset is loaded directly from `sklearn.datasets` (no CSV needed) and wrapped into a dataframe so it matches the same `df, target_column` pattern as the earlier assignments.

In [ ]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df["diagnosis"] = data.target   # 0 = malignant, 1 = benign

exp6_output = assn6_experiment(df, "diagnosis")

exp6_output["results_df"].to_csv("Experiment6_Results.csv", index=False)
exp6_output["results_df"]